# 02 - Sources and first models

**You will learn**: `source()`, `ref()`, models as `select` statements, materializations (table vs view),
`dbt run`, the `--select` syntax and the DAG.

**You will build**: the silver layer, one `stg_*` (staging) model per bronze table.

A staging model is the first, light transformation on a raw table: **rename, cast, trim, derive simple columns**.
No joins, no business logic yet.

> **dbt in the notebook or in the terminal.** Every `dbt("...")` cell prints, before its output, the **terminal equivalent** of the command.
> You can run any of them in a terminal instead: activate the virtual environment (from the repository root), then type the command shown.
> The result is exactly the same. (Make sure the files are saved: notebook cells that start with `%%writefile` write them for you.)

In [2]:
from helpers import *

## What is dbt?

dbt (data build tool) is the **T** of **ELT**: the raw data is already loaded in your warehouse,
dbt transforms it *inside* the warehouse with plain SQL `select` statements.

| You write | dbt does |
|---|---|
| a `select` statement per model (a `.sql` file) | wraps it in `create table / view ... as`, in the right order |
| `{{ ref('other_model') }}` | works out the dependency graph (DAG) |
| tests in YAML | runs them as SQL queries |
| descriptions in YAML | generates a documentation site with lineage |

## The architecture we will build (medallion)

```
   shared, loaded by                 dbt                                   dbt
   your trainer
 ┌───────────────────┐   ┌───────────────────────┐   ┌────────────────────────────────┐
 │ BRONZE            │   │ SILVER                │   │ GOLD                           │
 │ bronze.sports_shop│──▶│ silver.<your_schema>  │──▶│ gold.<your_schema>             │
 │ shared, read-only │   │ cleaned stg_* models  │   │ dim_*, fct_sales, mart_*       │
 │ raw, a bit dirty  │   │ seed: countries       │   └────────────────────────────────┘
 └───────────────────┘   │ snapshots (history)   │
                         └───────────────────────┘
```

| Layer | Who builds it | Goal |
|---|---|---|
| Bronze | your trainer (a generator notebook), **one copy shared by the whole class** | raw source-system data, kept as delivered; you only read it |
| Silver | **you**, with dbt | one clean, typed, deduplicated table per business entity |
| Gold | **you**, with dbt | star schema and KPI tables ready for analysts |

Everything you build lands in **your personal schema**, the `schema` of your dbt profile (for example `olivier_hurni`),
in the `silver` and `gold` catalogs. That way many people can work on the same bronze data without overwriting each other.

## 1. Declare the sources

A **source** tells dbt about a table that it does not build (our bronze tables).
Declaring it gives us `{{ source('bronze', 'products') }}` instead of a hard-coded table name, and puts the raw tables in the lineage graph.

Run the next cell: it writes the file `models/silver/_sources.yml` into your project. Read it before moving on.

In [ ]:
%%writefile ../../src/models/silver/_sources.yml
version: 2
sources:
- name: bronze
  description: Raw tables of the AlpSport source system, one row per record as delivered.
  database: bronze
  schema: sports_shop
  tables:
  - name: products
    description: Product catalog (current state, updated in place by the source system).
    columns:
    - name: product_id
  - name: stores
    description: The 11 physical stores and the online shop.
    columns:
    - name: store_id
  - name: sales_persons
    description: Sales persons and the store they currently work in.
    columns:
    - name: sales_person_id
  - name: customers
    description: Customers with their loyalty tier.
    columns:
    - name: customer_id
  - name: sales_orders
    description: Order headers. Contains duplicates, inconsistent status casing and orphan customer ids.
    columns:
    - name: order_id
    - name: status
    - name: customer_id
  - name: sales_order_lines
    description: Order lines. Contains duplicates, negative quantities and orphan product ids.
    columns:
    - name: order_line_id
    - name: quantity
    - name: product_id


Key points: `database: bronze` is the **catalog**, `schema: sports_shop` the schema. Each table has a name and a description.

In [ ]:
dbt("ls --select source:*")

## 2. Your first model: `stg_products`

A model is a file `models/<folder>/<name>.sql` containing **one `select`**. The file name is the model name.
`dbt run` turns it into `create table silver.<your_schema>.stg_products as (select ...)`.

**Exercise A.** Complete the model: select the listed columns explicitly (never `select *` in staging, so you control the contract)
and clean `product_name` with `trim()`. Run the cell to create the file, then edit it in the editor
(`src/models/silver/stg_products.sql`).

In [ ]:
%%writefile ../../src/models/silver/stg_products.sql
-- TODO 1: replace * by the columns:
--   product_id, product_name, category, brand, list_price, unit_cost, is_active, created_at, updated_at
-- TODO 2: trim the spaces around product_name
select
    *
from {{ source('bronze', 'products') }}


Before running it, look at what dbt will really send to Databricks. `dbt compile` renders the Jinja:

In [ ]:
dbt("compile --select stg_products")
show_file("target/compiled/alpsport/models/silver/stg_products.sql")

In [ ]:
dbt("run --select stg_products")

In [ ]:
check("stg_products exists with rows", scalar(f"SELECT COUNT(*) FROM silver.{SCHEMA}.stg_products") > 0)
q(f"SELECT * FROM silver.{SCHEMA}.stg_products LIMIT 5")

> **Stuck?** `restore_checkpoint(2)` at the end of this notebook gives you all the finished files.

**Exercise B.** `stg_stores`: same idea, plus a derived boolean column `is_online` that is true when `store_type = 'online'`.

In [ ]:
%%writefile ../../src/models/silver/stg_stores.sql
-- TODO: add the column is_online (true when store_type = 'online')
select
    store_id,
    store_name,
    city,
    canton,
    store_type,
    opened_date
from {{ source('bronze', 'stores') }}


The next two models are provided. Note the small transformations: `concat_ws` builds a full name, `lower`/`upper` normalise text.

In [ ]:
%%writefile ../../src/models/silver/stg_sales_persons.sql
select
    sales_person_id,
    first_name,
    last_name,
    concat_ws(' ', first_name, last_name) as full_name,
    lower(email) as email,
    store_id,
    hire_date,
    updated_at
from {{ source('bronze', 'sales_persons') }}


In [ ]:
%%writefile ../../src/models/silver/stg_customers.sql
select
    customer_id,
    first_name,
    last_name,
    concat_ws(' ', first_name, last_name) as full_name,
    lower(email) as email,
    city,
    upper(country_code) as country_code,
    loyalty_tier,
    created_at,
    updated_at
from {{ source('bronze', 'customers') }}


**Exercise C.** The order tables. In `stg_sales_orders` add `order_day`, the order date without the time
(`cast(order_date as date)`). `stg_sales_order_lines` is provided.

In [ ]:
%%writefile ../../src/models/silver/stg_sales_orders.sql
select
    order_id,
    order_date,
    -- TODO: add order_day = cast(order_date as date)
    customer_id,
    store_id,
    sales_person_id,
    channel,
    payment_method,
    status,
    created_at,
    updated_at,
    _batch_id
from {{ source('bronze', 'sales_orders') }}


In [ ]:
%%writefile ../../src/models/silver/stg_sales_order_lines.sql
select
    order_line_id,
    order_id,
    product_id,
    quantity,
    unit_price,
    discount_pct,
    _batch_id
from {{ source('bronze', 'sales_order_lines') }}


## 3. Run everything and read the DAG

`dbt run` builds all the models. dbt runs them in dependency order, in parallel where it can (`threads` in your profile).

In [ ]:
dbt("run")

### Selecting what to run

The `--select` option (`-s`) is the tool you will use all the time:

| Selector | Meaning |
|---|---|
| `--select stg_products` | one model |
| `--select stg_*` | wildcard |
| `--select +stg_sales_orders` | the model and everything **upstream** |
| `--select stg_sales_orders+` | the model and everything **downstream** |
| `--select tag:silver` | everything with the tag `silver` (set in `dbt_project.yml`) |
| `--exclude stg_stores` | everything except |

In [ ]:
dbt("ls --select tag:silver")

In [ ]:
dbt("ls --select +stg_sales_orders --output name")

## 4. Materializations: table vs view

`materialized: table` in `dbt_project.yml` makes every silver model a table. A model can override it with a `config` block.

**Exercise D.** Make `stg_stores` a **view**: add this as the first line of `stg_stores.sql`, then run it.

```sql
{{ config(materialized='view') }}
```

In [ ]:
q(f'''
    SELECT table_name, table_type
    FROM silver.information_schema.tables
    WHERE table_schema = '{SCHEMA}' ORDER BY table_name
''')

In [ ]:
dbt("run --select stg_stores")

In [ ]:
t = q(f"SELECT table_name, table_type FROM silver.information_schema.tables WHERE table_schema = '{SCHEMA}'")
check("stg_stores is now a view", t.set_index("table_name").loc["stg_stores", "table_type"] == "VIEW",
      "add the config block at the top of stg_stores.sql and re-run")
t

| Materialization | Use it when |
|---|---|
| `view` | light transformation, cheap to recompute, always fresh |
| `table` | heavy or often queried; rebuilt entirely on each run |
| `incremental` | big tables where only new rows should be processed (notebook 06) |
| `ephemeral` | a reusable CTE that never lands in the warehouse |

## 5. Something is off...

Compare the number of order rows with the number of distinct order ids in your new silver model.

In [ ]:
q(f'''
    SELECT COUNT(*) AS rows, COUNT(DISTINCT order_id) AS distinct_orders
    FROM silver.{SCHEMA}.stg_sales_orders
''')

There are more rows than orders: the bronze data contains **duplicates**, and our staging model happily passed them through.
Nothing in dbt told us so. In notebook 04 we will write **tests** that catch this, and fix the model.

## Recap

* `source()` points to raw tables, `ref()` to other models (see notebook 03).
* One model = one `select` in one file; dbt handles `create table` and ordering.
* `dbt run`, `dbt compile`, `dbt ls` and the `--select` syntax.
* Materializations control how a model is stored.

---
*Catch-up cell: run it only if you want to overwrite your work with the finished files of this module.*

In [ ]:
# restore_checkpoint(2)